# Outdated vs. Irrelevant: 20-item pilot

Reference year: **2020**. Source: the official SituatedQA temporal QA train and dev data. The selected rows are candidates; verify each dated answer in `data/fact_audit.csv` before collecting responses or presenting results. The notebook runs the classifier and evaluation when complete model-response and human-label CSVs are present.

In [ ]:
import csv, html, json, subprocess, sys
from pathlib import Path
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Open this notebook from the project or notebooks folder'

def read_rows(path):
    with path.open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

def show_table(rows, columns):
    cells = ['<table style="border-collapse:collapse;font-size:13px"><thead><tr>']
    cells += [f'<th style="border:1px solid #aaa;padding:5px">{html.escape(col)}</th>' for col in columns]
    cells.append('</tr></thead><tbody>')
    for row in rows:
        cells.append('<tr>')
        cells += [f'<td style="border:1px solid #aaa;padding:5px">{html.escape(str(row.get(col, "")))}</td>' for col in columns]
        cells.append('</tr>')
    cells.append('</tbody></table>')
    display(HTML(''.join(cells)))

questions = read_rows(ROOT / 'data/sample_questions.csv')
display(Markdown(f'## Selected questions ({len(questions)})'))
show_table(questions, ['id', 'question', 'current_answer', 'historical_answers'])

In [ ]:
responses = ROOT / 'data/model_responses.csv'
humans = ROOT / 'data/human_labels.csv'
if not responses.exists():
    display(Markdown('**Next step:** finish the fact audit, then collect SituatedQA answers into `data/model_responses.csv`.'))
else:
    response_rows = read_rows(responses)
    display(Markdown('## Collected model answers'))
    show_table(response_rows, ['id', 'model_answer'])

if not humans.exists():
    display(Markdown('**Next step:** annotate all 20 answers in `data/human_labels.csv` without viewing automatic labels.'))
elif responses.exists():
    subprocess.run([sys.executable, str(ROOT / 'src/evaluate.py')], cwd=ROOT, check=True)
    analysis = read_rows(ROOT / 'data/analysis_table.csv')
    metrics = json.loads((ROOT / 'results/metrics.json').read_text(encoding='utf-8'))
    matrix = read_rows(ROOT / 'results/confusion_matrix.csv')
    display(Markdown('## Paired analysis table with automatic and human labels'))
    show_table(analysis, ['id', 'model_answer', 'auto_label', 'human_label'])
    display(Markdown(f"## Metrics\n\nAgreement: **{metrics['agreement']:.1%}**; Cohen's kappa: **{metrics['cohens_kappa']}**; N = **{metrics['n']}**."))
    show_table([{'class': label, **values} for label, values in metrics['per_class'].items()], ['class', 'support', 'precision', 'recall', 'f1'])
    display(Markdown('## Confusion matrix (rows: human; columns: automatic)'))
    show_table(matrix, ['human\\auto', 'Correct', 'Outdated', 'Irrelevant'])
    display(Markdown('## One automatic example per class'))
    examples = [next((row for row in analysis if row['auto_label'] == label), None) for label in ('Correct', 'Outdated', 'Irrelevant')]
    show_table([row for row in examples if row], ['id', 'question', 'model_answer', 'auto_label', 'human_label'])
    if any(row is None for row in examples):
        display(Markdown('Some classes have no automatically labeled example in this sample.'))